In [ ]:
import kagglehub
path = kagglehub.dataset_download("patrickfleith/nasa-battery-dataset")

In [ ]:
# ============================================================
# BATTERY SOH ESTIMATION WITH TRANSFORMER + PHYSICS LOSS (SPM)
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm

# parameters and cutoff values
V_CUTOFF = 2.7
CAPACITY_MIN_AH = 1.0
NOMINAL_AH = 2.0
BINS = 20
M = 200

EPOCHS = 30
BATCH_SIZE = 64
LR = 1e-4

lambda_q = 0.3
lambda_e = 0.3
lambda_phys = 0.3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load Dataset

metadata = pd.read_csv(os.path.join(path, "metadata.csv"))
metadata["battery_id"] = metadata["battery_id"].astype(str)

excluded_batteries = ["B0049", "B0050", "B0051", "B0052"]

discharge_metadata = metadata[
    (metadata["type"] == "discharge") &
    (~metadata["battery_id"].isin(excluded_batteries))
].copy()

discharge_metadata["cycle_number"] = discharge_metadata.groupby("battery_id").cumcount() + 1

main_rows = []
phys_rows = []

# preprocessing
for _, row in tqdm(discharge_metadata.iterrows(), total=len(discharge_metadata)):
    file_path = os.path.join(path, "data", row["filename"])
    df = pd.read_csv(file_path).sort_values("Time").drop_duplicates("Time")

    if len(df) < 10:
        continue

    cutoff_idx = df[df["Voltage_measured"] < V_CUTOFF].index.min()
    if not pd.isna(cutoff_idx):
        df = df.loc[:cutoff_idx]

    if len(df) < 10:
        continue

    df["dt_hr"] = df["Time"].diff().fillna(0) / 3600
    df["dQ"] = df["Current_measured"] * df["dt_hr"]
    capacity = abs(df["dQ"].sum())

    if capacity <= CAPACITY_MIN_AH or capacity >= NOMINAL_AH:
        continue

    df["CumQ"] = df["dQ"].cumsum()
    df["SoC"] = 100 * (1 + df["CumQ"] / capacity)

    soh = 100 * capacity / NOMINAL_AH

    t = df["Time"].values
    t_grid = np.linspace(t[0], t[-1], M)
    dt_hr = np.diff(t_grid, prepend=t_grid[0]) / 3600

    def interp(col):
        return np.interp(t_grid, t, df[col])

    V = interp("Voltage_measured")
    I = interp("Current_measured")
    T = interp("Temperature_measured")
    SOC = interp("SoC")

    # Physics dataset
    for i in range(M):
        phys_rows.append({
            "t": t_grid[i] - t_grid[0],
            "dt_hr": dt_hr[i],
            "V": V[i],
            "I": I[i],
            "SOC": SOC[i],
            "SOH": soh
        })

    # Binned main dataset
    rs = pd.DataFrame({"V": V, "I": I, "T": T, "SOC": SOC, "dt_hr": dt_hr})
    chunks = np.array_split(rs, BINS)

    for b, c in enumerate(chunks):
        main_rows.append({
            "V": c["V"].mean(),
            "I": c["I"].mean(),
            "T": c["T"].mean(),
            "SOC": c["SOC"].mean(),
            "dt_hr": c["dt_hr"].sum(),
            "SOH": soh
        })

main_df = pd.DataFrame(main_rows)
phys_df = pd.DataFrame(phys_rows)

# datasets
X = main_df[["V", "I", "T", "SOC", "dt_hr"]].values.reshape(-1, BINS, 5)
y = main_df["SOH"].values.reshape(-1, 1)

X = torch.tensor(X, dtype=torch.float32).to(DEVICE)
y = torch.tensor(y, dtype=torch.float32).to(DEVICE)

# transformer
class SOHTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Linear(5, 64)
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=64, nhead=4, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=2)
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        x = self.embed(x)
        attn_out = self.encoder(x)
        pooled = attn_out.mean(dim=1)
        return self.head(pooled), attn_out

model = SOHTransformer().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
mse = nn.MSELoss()

# trainig data

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    soh_pred, attn = model(X)

    # DATA LOSS
    L_data = mse(soh_pred, y)

    # COULOMB CONSTRAINT
    Q_pred = torch.sum(X[:,:,1] * X[:,:,4], dim=1, keepdim=True)
    Q_true = y / 100 * NOMINAL_AH
    L_q = mse(Q_pred, Q_true)

    # ENERGY CONSTRAINT
    E_pred = torch.sum(X[:,:,0] * X[:,:,1] * X[:,:,4], dim=1, keepdim=True)
    E_true = E_pred.detach()
    L_e = mse(E_pred, E_true)

    loss = L_data + lambda_q*L_q + lambda_e*L_e
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    print(f"Epoch {epoch:02d} | Loss {loss.item():.4f}")

# evaluation
model.eval()
with torch.no_grad():
    y_hat, attn = model(X)

y_hat = y_hat.cpu().numpy()
y_true = y.cpu().numpy()

# visualation

# 1. Pred vs True
plt.figure()
plt.scatter(y_hat, y_true, s=5)
plt.plot([0,100],[0,100])
plt.xlabel("Predicted SoH")
plt.ylabel("True SoH")
plt.title("Prediction vs Truth")
plt.show()

# 2. Residual Histogram
res = y_hat - y_true
plt.figure()
plt.hist(res, bins=50)
plt.title("Residual Distribution")
plt.xlabel("Pred - True")
plt.show()

# 3. Attention Heatmap (mean)
attn_mean = attn.mean(dim=0).cpu().numpy()

plt.figure()
plt.imshow(attn_mean, aspect="auto")
plt.colorbar()
plt.title("Mean Attention Weights")
plt.xlabel("Key positions")
plt.ylabel("Query positions")
plt.show()

# 4. Loss curve
plt.figure()
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

print("Training complete.")
